# Oil Price Change Point Detection - Interactive Notebook

This notebook provides a simplified, step-by-step workflow for detecting regime shifts in oil prices using Bayesian change point analysis.

---

## Setup

Import all necessary modules and configure display settings.

In [ ]:
# Import modules
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys

# Add src to path if running from notebooks directory and package is not installed
sys.path.append(str(Path.cwd().parent / 'src'))

# Import our custom modules
from oil_price_changepoint.data import load_oil_data, clean_data, prepare_data_for_modeling
from oil_price_changepoint.eda import calculate_statistics, plot_time_series, plot_distribution, plot_volatility
from oil_price_changepoint.model import (build_changepoint_model, sample_model, 
                                        check_convergence, extract_results)
from oil_price_changepoint.visualization import plot_changepoint_results, plot_trace_diagnostics, plot_regime_comparison
from oil_price_changepoint.report import generate_report


plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(" All modules imported successfully!")

---

## Step 1: Load and Prepare Data

Load the oil price dataset and prepare it for analysis.

In [ ]:
# Define data path
data_path = Path('../data/BrentOilPrices.csv')

# Load data
print("Loading oil price data...")
if not data_path.exists():
    print(f"Error: Data file not found at {data_path.absolute()}")
else:
    df = load_oil_data(data_path)
    print(f"✓ Loaded {len(df)} records\n")

    # Display first few rows
    print("First 5 rows:")
    display(df.head())

In [ ]:
# Clean data
print("Cleaning data...")
df_clean = clean_data(df)
print(f"✓ Cleaned data: {len(df_clean)} records")

# Prepare for modeling
print("\nPreparing data for modeling...")
data_dict = prepare_data_for_modeling(df_clean)

print(f"\n{'='*50}")
print("DATA SUMMARY")
print(f"{'='*50}")
print(f"Observations: {data_dict['n_days']}")
print(f"Date range: {data_dict['dates'][0]} to {data_dict['dates'][-1]}")
print(f"Price range: ${data_dict['oil_prices'].min():.2f} - ${data_dict['oil_prices'].max():.2f}")
print(f"Mean: ${data_dict['historical_mean']:.2f}")
print(f"Std Dev: ${data_dict['historical_std']:.2f}")
print(f"{'='*50}")

---

## Step 2: Exploratory Data Analysis

Visualize the data to understand trends, distributions, and volatility patterns.

In [ ]:
# Calculate statistics
stats = calculate_statistics(df_clean)

In [ ]:
# Plot time series
print("Time Series Plot:")
plot_time_series(df_clean)

In [ ]:
# Plot distribution
print("Price Distribution:")
plot_distribution(df_clean)

In [ ]:
# Plot volatility
print("Volatility Analysis:")
plot_volatility(df_clean, window=30)

---

## Step 3: Build Bayesian Change Point Model

Construct the Bayesian model with appropriate priors and likelihood.

In [ ]:
print("Building Bayesian change point model...\n")

model = build_changepoint_model(
    data_dict['oil_prices'],
    data_dict['historical_mean'],
    data_dict['historical_std']
)

print("✓ Model built successfully!")
print("\nModel Structure:")
print("  - Change point (τ): DiscreteUniform(0, n_days-1)")
print("  - Price before (μ₁): Normal(historical_mean, 20)")
print("  - Price after (μ₂): Normal(historical_mean, 20)")
print("  - Volatility before (σ₁): HalfNormal(10)")
print("  - Volatility after (σ₂): HalfNormal(10)")

---

## Step 4: Run MCMC Sampling

Sample from the posterior distribution using MCMC.

**Note:** This may take 2-5 minutes depending on your hardware.

In [ ]:
# Run MCMC sampling
trace = sample_model(
    model, 
    draws=2000,      # Number of samples per chain
    tune=1000,       # Tuning iterations
    chains=4,        # Number of chains
    random_seed=42   # For reproducibility
)

---

## Step 5: Check Convergence

Verify that MCMC sampling converged properly.

In [ ]:
# Check convergence
convergence_ok = check_convergence(trace)

if convergence_ok:
    print("\n✅ MCMC sampling converged successfully!")
    print("Results are reliable and can be interpreted with confidence.")
else:
    print("\n⚠️ WARNING: Convergence issues detected!")
    print("Consider increasing draws or tune parameters.")

---

## Step 6: Extract Results

Extract the detected change point and price level estimates.

In [ ]:
# Extract results
results = extract_results(trace, data_dict['dates'])

In [ ]:
# Display key findings
print("\n" + "="*60)
print(" "*15 + "KEY FINDINGS")
print("="*60)

print(f"\n🎯 DETECTED CHANGE POINT: {results['change_date']}")
print(f"\n📊 95% Credible Interval:")
print(f"   {results['date_lower']} to {results['date_upper']}")

print(f"\n💰 PRICE ANALYSIS:")
print(f"   Before: ${results['mu_before_mean']:.2f} ± ${results['mu_before_std']:.2f}")
print(f"   After:  ${results['mu_after_mean']:.2f} ± ${results['mu_after_std']:.2f}")
print(f"   Change: ${results['price_change']:.2f} ({results['price_change_pct']:+.1f}%)")

print(f"\n📈 VOLATILITY:")
print(f"   Before: ${results['sigma_before_mean']:.2f}")
print(f"   After:  ${results['sigma_after_mean']:.2f}")

print("\n" + "="*60)

---

## Step 7: Visualize Results

Create comprehensive visualizations of the detected change point.

In [ ]:
# Main results dashboard
print("Change Point Detection Results:")
plot_changepoint_results(
    data_dict['dates'],
    data_dict['oil_prices'],
    results,
    trace
)

In [ ]:
# Trace diagnostics
print("MCMC Trace Diagnostics:")
plot_trace_diagnostics(trace)

In [ ]:
# Regime comparison
print("Regime Comparison:")
plot_regime_comparison(
    data_dict['dates'],
    data_dict['oil_prices'],
    results
)

---

## Step 8: Generate Report

Create a comprehensive markdown report with all findings.

In [ ]:
# Create outputs directory
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# Generate report
report_path = output_dir / 'analysis_report.md'
report_text = generate_report(results, data_dict, convergence_ok, report_path)

print("\n✓ Report generated successfully!")
print(f"\n📄 Report saved to: {report_path.absolute()}")

---

## Step 9: Interpretation

Interpret the results in context of historical events.

In [ ]:
# Interpretation helper
change_date = results['change_date']
change_year = change_date.astype('datetime64[Y]').astype(int) + 1970
change_month = change_date.astype('datetime64[M]').astype(int) % 12 + 1

print("\n" + "="*60)
print(" "*20 + "INTERPRETATION")
print("="*60)

print(f"\nDetected change point: {change_date}")
print(f"Year: {change_year}, Month: {change_month}")

# Check for known events
if change_year == 2020 and change_month in [2, 3, 4]:
    print("\n🔍 HISTORICAL CONTEXT:")
    print("   This aligns with the COVID-19 pandemic onset (early 2020).")
    print("   The pandemic caused a dramatic collapse in oil demand and prices.")
    print("   Oil prices fell from ~$65 to below $25 in March-April 2020.")

print("\n💡 RECOMMENDATIONS:")
print("   1. Validate against historical oil market events")
print("   2. Consider geopolitical factors (OPEC, wars, sanctions)")
print("   3. Analyze supply/demand dynamics around the change point")
print("   4. Use detected regimes for forecasting future prices")
print("\n" + "="*60)

---

## Summary

### What We Accomplished:

1. ✅ Loaded and cleaned oil price data
2. ✅ Performed exploratory data analysis
3. ✅ Built Bayesian change point model
4. ✅ Ran MCMC sampling with convergence checks
5. ✅ Detected regime shift with credible intervals
6. ✅ Generated comprehensive visualizations
7. ✅ Created detailed analysis report

### Next Steps:

- Review the generated report in `outputs/analysis_report.md`
- Validate findings against historical events
- Try with your own oil price data
- Extend to multiple change points if needed

---

## Optional: Save All Visualizations

Run this cell to save all plots to the outputs directory.

In [ ]:
# Save all visualizations
print("Saving all visualizations...\n")

# EDA plots
plot_time_series(df_clean, save_path=output_dir / 'eda_time_series.png')
plot_distribution(df_clean, save_path=output_dir / 'eda_distribution.png')
plot_volatility(df_clean, window=30, save_path=output_dir / 'eda_volatility.png')

# Results plots
plot_changepoint_results(
    data_dict['dates'],
    data_dict['oil_prices'],
    results,
    trace,
    save_path=output_dir / 'changepoint_results.png'
)

plot_trace_diagnostics(
    trace,
    save_path=output_dir / 'trace_diagnostics.png'
)

plot_regime_comparison(
    data_dict['dates'],
    data_dict['oil_prices'],
    results,
    save_path=output_dir / 'regime_comparison.png'
)

print("\n✓ All visualizations saved to:", output_dir.absolute())